# CrewAI Hybrid Campaign Planner (ROI-Driven)

This notebook upgrades the earlier design by **removing `customer_review` from prediction logic** and using **business KPIs** instead:

- `buy_prob`
- `spend_propensity_score`
- `spend_uplift_score`
- `return_prob`
- `recommend_prob`
- expected revenue
- expected incremental profit
- ROI

The pipeline is intentionally **hybrid**:

1. **Deterministic Python logic** computes commercial signals and month-level campaign candidates.
2. **Optimization** chooses the most effective month under budget and resource constraints.
3. **CrewAI agents** explain the outcome in business language.

> Agent 1 does **not** predict free-form reviews anymore. It produces a structured **commercial opportunity summary**.
> Agent 2 recommends the best campaign month using the optimized month-level inputs.

## Cell 1: Install dependencies

Run this cell once in a clean environment.  
If you already installed the packages, you can skip it.

In [ ]:
# %pip is safer inside notebooks than !pip
%pip install -q crewai pulp pandas numpy python-dotenv

## Cell 2: Import libraries and define global assumptions

These assumptions make the notebook business-friendly:
- `profit_margin_rate`: gross margin proxy
- `campaign_cost_rate`: variable campaign cost as % of retail value
- `fixed_monthly_campaign_cost`: base cost of running a monthly campaign

In [ ]:
import os
import json
from datetime import date, datetime

import numpy as np
import pandas as pd
from pulp import LpBinary, LpMaximize, LpProblem, LpVariable, lpSum, value

# Optional CrewAI imports
from crewai import Agent, Crew, Process, Task

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Business assumptions (edit if needed)
profit_margin_rate = 0.22              # gross margin assumption
campaign_cost_rate = 0.05              # variable campaign cost as % of online retail value
fixed_monthly_campaign_cost = 25000    # fixed cost to run one month of campaign
quarterly_budget = 180000              # total budget available across candidate months
max_selected_months = 1                # set to 1 for single best month selection
max_resource_utilization = 0.85        # monthly operational cap

## Cell 3: Load the uploaded sample dataset

This cell looks for the uploaded CSV in the notebook environment first.

In [ ]:
possible_paths = [
    "/mnt/data/customer_behavior_with_reviews_scores.csv",
    "customer_behavior_with_reviews_scores.csv"
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Could not find customer_behavior_with_reviews_scores.csv")

raw_df = pd.read_csv(data_path)
print(f"Loaded data from: {data_path}")
print(raw_df.shape)
raw_df.head(3)

## Cell 4: Validate columns and standardize data types

We only require business-relevant model outputs and commercial fields.

In [ ]:
required_cols = [
    "cust_id",
    "date",
    "online_retail_value",
    "buy_prob",
    "spend_propensity_score",
    "spend_uplift_score",
    "return_prob",
    "recommend_prob"
]

missing = [c for c in required_cols if c not in raw_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work_df = raw_df.copy()

# Parse dates
work_df["date"] = pd.to_datetime(work_df["date"], errors="coerce")
if work_df["date"].isna().any():
    bad_rows = work_df[work_df["date"].isna()].shape[0]
    print(f"Warning: {bad_rows} rows have invalid date and will be dropped.")
    work_df = work_df.dropna(subset=["date"]).copy()

# Numeric coercion
numeric_cols = [
    "online_retail_value",
    "buy_prob",
    "spend_propensity_score",
    "spend_uplift_score",
    "return_prob",
    "recommend_prob"
]

for col in numeric_cols:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

work_df = work_df.dropna(subset=numeric_cols).copy()

# Clip probabilities to [0, 1]
prob_cols = ["buy_prob", "spend_propensity_score", "return_prob", "recommend_prob"]
for col in prob_cols:
    work_df[col] = work_df[col].clip(0, 1)

# spend_uplift_score may exceed 1 depending on modeling convention; keep non-negative
work_df["spend_uplift_score"] = work_df["spend_uplift_score"].clip(lower=0)

print(work_df.shape)
work_df[required_cols].head(3)

## Cell 5: Build ROI-driven commercial signals

This is the key improvement.

We do **not** use `customer_review` at all.

Instead we derive:
- `expected_base_revenue`
- `expected_uplift_revenue`
- `expected_total_revenue`
- `expected_incremental_profit`
- `estimated_campaign_cost`
- `expected_roi`
- `commercial_priority_band`
- `predicted_future_value_summary`

In [ ]:
def add_roi_signals(df: pd.DataFrame) -> pd.DataFrame:
    temp = df.copy()

    # Base revenue expected from purchase probability and spend propensity
    temp["expected_base_revenue"] = (
        temp["online_retail_value"] *
        temp["buy_prob"] *
        temp["spend_propensity_score"]
    )

    # Incremental revenue expected from uplift if targeted
    temp["expected_uplift_revenue"] = (
        temp["expected_base_revenue"] *
        temp["spend_uplift_score"]
    )

    # Penalize total revenue by return probability
    temp["expected_total_revenue"] = (
        (temp["expected_base_revenue"] + temp["expected_uplift_revenue"]) *
        (1 - temp["return_prob"])
    )

    # Expected incremental profit contribution
    temp["expected_incremental_profit"] = (
        temp["expected_total_revenue"] * profit_margin_rate
    )

    # Per-customer campaign cost assumption
    temp["estimated_campaign_cost"] = (
        temp["online_retail_value"] * campaign_cost_rate
    )

    # ROI = profit / cost
    temp["expected_roi"] = np.where(
        temp["estimated_campaign_cost"] > 0,
        temp["expected_incremental_profit"] / temp["estimated_campaign_cost"],
        0
    )

    # Business-friendly prioritization
    conditions = [
        (temp["expected_roi"] >= 3.0) & (temp["return_prob"] < 0.20),
        (temp["expected_roi"] >= 1.5) & (temp["return_prob"] < 0.35),
        (temp["expected_roi"] >= 0.75)
    ]
    choices = ["High", "Medium", "Low"]
    temp["commercial_priority_band"] = np.select(conditions, choices, default="Deprioritize")

    # Agent 1 will summarize opportunity, but we also keep a deterministic textual label
    temp["predicted_future_value_summary"] = np.select(
        [
            temp["commercial_priority_band"].eq("High"),
            temp["commercial_priority_band"].eq("Medium"),
            temp["commercial_priority_band"].eq("Low")
        ],
        [
            "High-value target with strong expected ROI and manageable return risk.",
            "Promising target with positive ROI; suitable for selective campaign investment.",
            "Marginal target with limited upside; use lower-cost or controlled outreach."
        ],
        default="Low commercial attractiveness; avoid allocating primary campaign budget."
    )

    return temp

work_df = add_roi_signals(work_df)

signal_cols = [
    "cust_id",
    "date",
    "online_retail_value",
    "buy_prob",
    "spend_propensity_score",
    "spend_uplift_score",
    "return_prob",
    "recommend_prob",
    "expected_total_revenue",
    "expected_incremental_profit",
    "estimated_campaign_cost",
    "expected_roi",
    "commercial_priority_band",
    "predicted_future_value_summary"
]

work_df[signal_cols].head(10)

## Cell 6: Create month-level campaign summary

Optimization should operate at the **month** level, not individual customer rows.

In [ ]:
work_df["campaign_month"] = work_df["date"].dt.to_period("M").astype(str)

monthly_summary = (
    work_df.groupby("campaign_month", as_index=False)
    .agg(
        customer_count=("cust_id", "count"),
        total_expected_revenue=("expected_total_revenue", "sum"),
        total_expected_profit=("expected_incremental_profit", "sum"),
        total_estimated_campaign_cost=("estimated_campaign_cost", "sum"),
        avg_buy_prob=("buy_prob", "mean"),
        avg_spend_propensity=("spend_propensity_score", "mean"),
        avg_spend_uplift=("spend_uplift_score", "mean"),
        avg_return_prob=("return_prob", "mean"),
        avg_recommend_prob=("recommend_prob", "mean"),
        high_priority_customers=("commercial_priority_band", lambda s: (s == "High").sum()),
        medium_priority_customers=("commercial_priority_band", lambda s: (s == "Medium").sum())
    )
)

monthly_summary["month_level_roi"] = np.where(
    monthly_summary["total_estimated_campaign_cost"] > 0,
    monthly_summary["total_expected_profit"] / monthly_summary["total_estimated_campaign_cost"],
    0
)

# Approximate operational load: more customers + more high-priority segments => more effort
monthly_summary["resource_utilization"] = (
    0.50 * (monthly_summary["customer_count"] / monthly_summary["customer_count"].max()) +
    0.30 * (monthly_summary["high_priority_customers"] / monthly_summary["high_priority_customers"].max()) +
    0.20 * (monthly_summary["avg_spend_uplift"] / monthly_summary["avg_spend_uplift"].max())
).clip(0, 1)

# Add a fixed cost per active campaign month
monthly_summary["all_in_campaign_cost"] = (
    monthly_summary["total_estimated_campaign_cost"] + fixed_monthly_campaign_cost
)

# Objective score for optimization
monthly_summary["campaign_objective_score"] = (
    monthly_summary["total_expected_profit"] *
    (1 + 0.20 * monthly_summary["avg_recommend_prob"]) *
    (1 - 0.50 * monthly_summary["avg_return_prob"])
)

monthly_summary.sort_values(["campaign_objective_score", "month_level_roi"], ascending=False).head(12)

## Cell 7: Optimize the best campaign month under budget and capacity

This chooses the best month using:
- budget
- operational load
- ROI / profit-driven objective

You can set `max_selected_months = 1` or more at the top.

In [ ]:
model = LpProblem("ROI_Driven_Campaign_Selection", LpMaximize)

months = monthly_summary["campaign_month"].tolist()

x = {
    m: LpVariable(f"select_{m}", cat=LpBinary)
    for m in months
}

objective_lookup = dict(zip(monthly_summary["campaign_month"], monthly_summary["campaign_objective_score"]))
cost_lookup = dict(zip(monthly_summary["campaign_month"], monthly_summary["all_in_campaign_cost"]))
resource_lookup = dict(zip(monthly_summary["campaign_month"], monthly_summary["resource_utilization"]))

# Objective: maximize overall campaign score
model += lpSum(objective_lookup[m] * x[m] for m in months)

# Select up to N months
model += lpSum(x[m] for m in months) <= max_selected_months

# Total budget cap
model += lpSum(cost_lookup[m] * x[m] for m in months) <= quarterly_budget

# Resource cap
model += lpSum(resource_lookup[m] * x[m] for m in months) <= max_resource_utilization

status = model.solve()

selected_months = [m for m in months if x[m].value() == 1]
selected_months

## Cell 8: Inspect optimized plan

This is the deterministic business recommendation that Agent 2 will explain.

In [ ]:
optimized_plan_df = (
    monthly_summary[monthly_summary["campaign_month"].isin(selected_months)]
    .sort_values(["campaign_objective_score", "month_level_roi"], ascending=False)
    .reset_index(drop=True)
)

if optimized_plan_df.empty:
    print("No month was selected under the current budget/resource constraints.")
else:
    display(optimized_plan_df)

## Cell 9: Inspect top high-value customer opportunities

This is the data Agent 1 can narrate.

In [ ]:
top_targets_df = (
    work_df.sort_values(
        ["expected_roi", "expected_incremental_profit", "recommend_prob"],
        ascending=False
    )[
        [
            "cust_id",
            "campaign_month",
            "online_retail_value",
            "buy_prob",
            "spend_propensity_score",
            "spend_uplift_score",
            "return_prob",
            "recommend_prob",
            "expected_total_revenue",
            "expected_incremental_profit",
            "estimated_campaign_cost",
            "expected_roi",
            "commercial_priority_band",
            "predicted_future_value_summary"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

top_targets_df

## Cell 10: Helper to sanitize inputs for CrewAI

CrewAI prompt interpolation fails if you pass `Timestamp`, NumPy scalars, or NaN directly.  
This helper converts them to safe Python/JSON types.

In [ ]:
def sanitize_for_crewai(obj):
    if isinstance(obj, dict):
        return {str(k): sanitize_for_crewai(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_for_crewai(v) for v in obj]
    if isinstance(obj, tuple):
        return [sanitize_for_crewai(v) for v in obj]
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj

## Cell 11: Define prompt templates for both agents

### Agent 1 — Solution Analyst
Role:
- Convert model outputs into a commercial opportunity summary.
- No free-form review prediction.
- Focus on ROI, uplift, risk, and recommendation likelihood.

### Agent 2 — Product Owner
Role:
- Examine the optimized month-level plan.
- Recommend the most effective campaign month under constraints.

In [ ]:
solution_analyst_prompt_template = '''
You are the Solution Analyst.

You are given sample high-value customer records:
{top_targets_sample}

Your job:
1. Analyze expected ROI, expected incremental profit, recommendation probability,
   return risk, and spend uplift.
2. Explain what kind of customers should be prioritized for the next quarter campaign.
3. Do NOT use customer reviews.
4. Produce a structured commercial opportunity summary.

Return STRICT JSON:
{{
  "priority_customer_profile": "...",
  "key_value_drivers": ["...", "...", "..."],
  "main_risks": ["...", "..."],
  "recommended_action": "...",
  "summary": "..."
}}
'''

product_owner_prompt_template = '''
You are the Product Owner.

You are given the optimized month-level campaign plan:
{optimized_campaign_months}

Your job:
1. Evaluate ROI, expected profit, campaign cost, resource utilization, and return risk.
2. Recommend the best month to run the campaign.
3. Explain trade-offs clearly for business stakeholders.

Return STRICT JSON:
{{
  "recommended_month": "...",
  "why_this_month": "...",
  "budget_commentary": "...",
  "capacity_commentary": "...",
  "risk_commentary": "...",
  "final_recommendation": "..."
}}
'''

## Cell 12: Create CrewAI agents and tasks

> This section requires an LLM provider to be configured in your environment.
> For example, set `OPENAI_API_KEY` before running.

If you only want the deterministic business output, you can skip Cells 12–14.

In [ ]:
solution_analyst_agent = Agent(
    role="Solution Analyst",
    goal="Translate customer-level model outputs into commercial opportunity insights using ROI, uplift, and risk signals.",
    backstory=(
        "You are a commercial analytics specialist who converts predictive outputs into "
        "clear prioritization logic for business teams. You avoid vague narration and focus "
        "on monetization potential, risk, and campaign readiness."
    ),
    verbose=True
)

product_owner_agent = Agent(
    role="Product Owner",
    goal="Recommend the most effective campaign month under budget and resource constraints.",
    backstory=(
        "You are responsible for quarterly campaign planning. You prioritize months that "
        "maximize commercial impact while remaining feasible under cost and operational limits."
    ),
    verbose=True
)

solution_task = Task(
    description=solution_analyst_prompt_template,
    expected_output="Strict JSON describing the priority customer profile and campaign action summary.",
    agent=solution_analyst_agent
)

product_owner_task = Task(
    description=product_owner_prompt_template,
    expected_output="Strict JSON recommending the best campaign month with cost, capacity, and risk commentary.",
    agent=product_owner_agent
)

## Cell 13: Build Crew inputs

We pass:
- top prioritized customer sample
- optimized month-level campaign plan

In [ ]:
top_targets_sample = sanitize_for_crewai(top_targets_df.head(5).to_dict(orient="records"))
optimized_campaign_months = sanitize_for_crewai(optimized_plan_df.to_dict(orient="records"))

crew_inputs = {
    "top_targets_sample": top_targets_sample,
    "optimized_campaign_months": optimized_campaign_months
}

crew_inputs

## Cell 14: Run the Crew

This is optional.  
If your LLM provider is not configured, this cell may fail.  
The deterministic outputs from previous cells remain valid even if you skip this step.

In [ ]:
crew = Crew(
    agents=[solution_analyst_agent, product_owner_agent],
    tasks=[solution_task, product_owner_task],
    process=Process.sequential,
    verbose=True
)

try:
    crew_result = crew.kickoff(inputs=crew_inputs)
    print("Crew execution completed.\n")
    print(crew_result)
except Exception as e:
    print("Crew execution failed.")
    print("Deterministic optimization outputs are still available.")
    print("Error:", str(e))

## Cell 15: Export outputs

This writes the improved ROI-driven outputs to disk.

In [ ]:
work_df.to_csv("roi_driven_customer_opportunities.csv", index=False)
monthly_summary.to_csv("roi_driven_monthly_summary.csv", index=False)
optimized_plan_df.to_csv("optimized_campaign_plan.csv", index=False)

if "crew_result" in globals():
    with open("crew_agent_output.txt", "w", encoding="utf-8") as f:
        f.write(str(crew_result))

print("Export complete:")
print("- roi_driven_customer_opportunities.csv")
print("- roi_driven_monthly_summary.csv")
print("- optimized_campaign_plan.csv")
if "crew_result" in globals():
    print("- crew_agent_output.txt")